In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version built with PyTorch: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("PyTorch was not built with CUDA support or CUDA is not configured correctly.")


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*PossibleUserWarning.*")
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=".*to enable TensorBoard support.*")

import os, sys
sys.stderr = open(os.devnull, 'w')

In [ ]:
import sys
from pathlib import Path
from pathlib import Path

_cwd = Path().resolve()
_candidates = (_cwd, _cwd.parent)
for _root in _candidates:
    if (_root / 'eqm_decompositional_graph_generator').exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
else:
    raise ModuleNotFoundError("Could not locate 'eqm_decompositional_graph_generator' package; adjust sys.path.")

del _cwd, _candidates, _root


# Code

In [ ]:
# Shared notebook helpers (kept in a .py module to keep notebooks lean)
try:
    from notebooks.notebook_utils import (
        offset_neg_graphs,
        plot_sample,
        select_pos_neg,
        plot_networkx_graphs,
        show_samples,
        graph_label_histogram,
        summarize_label_histogram_alignment,
        plot_label_histogram_alignment,
        fit_graph_generator,
        run_label_histogram_analysis,
        compare_real_vs_generated,
    )
except ModuleNotFoundError:
    from notebook_utils import (
        offset_neg_graphs,
        plot_sample,
        select_pos_neg,
        plot_networkx_graphs,
        show_samples,
        graph_label_histogram,
        summarize_label_histogram_alignment,
        plot_label_histogram_alignment,
        fit_graph_generator,
        run_label_histogram_analysis,
        compare_real_vs_generated,
    )


In [ ]:
# Helpers moved to notebooks/notebook_utils.py


In [ ]:
# Helpers moved to notebooks/notebook_utils.py


In [ ]:
# Helpers moved to notebooks/notebook_utils.py


In [ ]:
# Helpers moved to notebooks/notebook_utils.py


In [ ]:
# Helpers moved to notebooks/notebook_utils.py


---

# Data

In [ ]:
DATASET_TYPE = 'ARTIFICIAL'  # Uncomment to use the synthetic dataset instead.

In [ ]:
DATASET_TYPE = 'MOLECULAR'

In [ ]:
from functools import partial

try:
    from notebooks.demo_pipeline_utils import build_dataset as _build_dataset
except ModuleNotFoundError:
    from demo_pipeline_utils import build_dataset as _build_dataset

# Dataset builder bound to notebook dataset type
build_dataset = partial(_build_dataset, dataset_type=DATASET_TYPE)


---

# Experiment

In [ ]:
import sys
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().parents[:2]]:
    if (_root / 'eqm_decompositional_graph_generator').exists():
        _root_str = str(_root.resolve())
        if _root_str not in sys.path:
            sys.path.insert(0, _root_str)
        break
else:
    raise ModuleNotFoundError("Could not locate 'eqm_decompositional_graph_generator' package from current working directory.")

del _root, _root_str

from sklearn.model_selection import train_test_split

from NSPPK.nsppk import NSPPK, NodeNSPPK
from AbstractGraph.abstract_graph_operators import compose, cycle, neighborhood, unlabel
from AbstractGraph.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
    WithinRangeFeasibilityEstimatorFromNumericalFunction,
)
from eqm_decompositional_graph_generator.node_engine import EqMDecompositionalNodeGenerator
from eqm_decompositional_graph_generator.graph_engine import (
    EqMDecompositionalGraphDecoder,
    EqMDecompositionalGraphGenerator,
)
from eqm_decompositional_graph_generator.graph_engine import sample_positive_endpoint_pair

try:
    from notebooks.notebook_utils import fit_graph_generator, run_label_histogram_analysis, compare_real_vs_generated
except ModuleNotFoundError:
    from notebook_utils import fit_graph_generator, run_label_histogram_analysis, compare_real_vs_generated

REPO_ROOT = next(
    candidate.resolve()
    for candidate in [Path.cwd(), Path.cwd().parent]
    if (candidate / 'eqm_decompositional_graph_generator').exists()
)
ARTIFACT_ROOT = REPO_ROOT / '.artifacts'
CHECKPOINT_ROOT = ARTIFACT_ROOT / 'checkpoints' / 'eqm'

MODEL_FILENAME = 'graph_generator_eqm.pkl'
MODEL_PATH = ARTIFACT_ROOT / MODEL_FILENAME


def save_graph_generator(graph_generator, filename=None):
    model_path = Path(filename) if filename is not None else MODEL_PATH
    model_path.parent.mkdir(parents=True, exist_ok=True)
    graph_generator.save(str(model_path))
    print(f"Saved graph generator to {model_path}")
    return model_path


def load_graph_generator(filename=None):
    model_path = Path(filename) if filename is not None else MODEL_PATH
    if not model_path.exists():
        raise FileNotFoundError(f"Graph generator file not found: {model_path}")
    loaded = EqMDecompositionalGraphGenerator().load(str(model_path))
    print(f"Loaded graph generator from {model_path}")
    return loaded



from functools import partial
try:
    from notebooks.demo_pipeline_utils import prepare_experiment as _prepare_experiment, build_graph_generator as _build_graph_generator
except ModuleNotFoundError:
    from demo_pipeline_utils import prepare_experiment as _prepare_experiment, build_graph_generator as _build_graph_generator

# Experiment preparation helper bound to notebook dataset function
prepare_experiment = partial(_prepare_experiment, build_dataset_fn=build_dataset)

# Generator builder bound to notebook artifact/checkpoint roots
build_graph_generator = partial(_build_graph_generator, artifact_root=ARTIFACT_ROOT, checkpoint_root=CHECKPOINT_ROOT, locality_horizon=1)


---

In [ ]:
# build_graph_generator moved to notebooks/demo_pipeline_utils.py


In [ ]:
graphs, targets, train_graphs, test_graphs, train_targets, test_targets = prepare_experiment(
    dataset_size=2000,
    test_size=9,
    random_state=42,
    size=27
)
graph_generator = build_graph_generator(nbits=11, verbose=2)


In [ ]:
%%time
graph_generator = fit_graph_generator(graph_generator, train_graphs)


In [ ]:
# Persist the trained generator
save_graph_generator(graph_generator)



In [ ]:
# Recover a graph generator from disk (uncomment to use)
# graph_generator = load_graph_generator()



## Conditioning vs Generated Label Histograms

In [ ]:
label_histogram_summary = run_label_histogram_analysis(graph_generator, train_graphs, n_compare=5)


In [ ]:
show_samples(n_samples=7, graphs=train_graphs, targets=train_targets, graph_generator=graph_generator)

In [ ]:
show_samples(n_samples=7, graphs=test_graphs, targets=test_targets, graph_generator=graph_generator)

## Positive-to-Positive Conditioning Interpolation

Select two positive training graphs at random, interpolate between them, and display the endpoints plus feasible generated graphs.

In [ ]:
selected_indices, selected_targets, endpoint_graph_a, endpoint_graph_b = sample_positive_endpoint_pair(
    train_graphs,
    train_targets,
)
print("Endpoint indices:", selected_indices)
print("Endpoint targets:", selected_targets)

display_mode = 'molecule' if DATASET_TYPE == 'MOLECULAR' else 'not_molecule'
print("Selected endpoint graphs:")
plot_networkx_graphs([endpoint_graph_a, endpoint_graph_b], n_cols=2, mode=display_mode)

interpolation_result = graph_generator.interpolate(
    endpoint_graph_a,
    endpoint_graph_b,
    k=3,
    apply_feasibility_filtering=True,
)
display(interpolation_result["summary"])

missing_steps = interpolation_result["summary"].loc[
    ~interpolation_result["summary"]["decoded"],
    "step",
].tolist()
if missing_steps:
    print("Interpolation steps rejected by feasibility filtering:", missing_steps)

graphs_to_display = [endpoint_graph_a, *interpolation_result["generated_graphs"], endpoint_graph_b]
print("Endpoints and generated interpolation graphs:")
plot_networkx_graphs(graphs_to_display, n_cols=max(1, len(graphs_to_display)), mode=display_mode)


## Real vs Generated Distribution Comparison

In [ ]:
distribution_comparison = compare_real_vs_generated(
    graph_generator,
    test_graphs,
    apply_feasibility_filtering=True,
)


---